# ⚾ 야구 자세 분석 - YOLO26 Pose 학습 (Google Colab)

AIHUB 야구 데이터(`Baseball_Sample`)를 YOLO Pose 형식으로 변환하고 **YOLO26 Pose** 모델을 학습합니다.

### 준비물
1. 로컬의 `Baseball_Sample` 폴더를 **zip으로 압축** → 구글 드라이브 `MyDrive/BROS_DATA/data/Baseball_Sample.zip` 에 업로드
   (폴더 안에 `라벨링데이터`(JSON), `원천데이터`(JPG)가 들어 있으면 됩니다)
2. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
3. 위에서부터 셀을 순서대로 실행

### 결과
학습 결과는 구글 드라이브 `MyDrive/BROS_DATA/runs/` 에 저장되고(런타임이 끊겨도 유지),
최종 모델은 `MyDrive/BROS_DATA/best_baseball_pose.pt` 로 복사됩니다. 이 파일을 `AI-Server`에 넣어 사용합니다.

## 1. 설정

In [ ]:
# ================= 사용자 설정 =================
DRIVE_DIR = "/content/drive/MyDrive/BROS_DATA"     # 구글 드라이브 작업 폴더 (학습 결과 저장 위치)
SAMPLE_ZIP = f"{DRIVE_DIR}/data/Baseball_Sample.zip"  # AIHUB 원본 압축파일
WORK_DIR = "/content"                              # 압축 해제/변환은 Colab 로컬 디스크에서 (드라이브보다 훨씬 빠름)

MODEL = "yolo26n-pose.pt"   # 모델 크기: n(가장 빠름) < s < m < l < x(가장 정확)
EPOCHS = 100                # 전체 데이터 반복 학습 횟수 (빠른 테스트는 10~20)
IMGSZ = 640                 # 입력 이미지 크기
BATCH = 16                  # GPU 메모리 부족(OOM) 시 8로 낮추기
PATIENCE = 20               # 이 epoch 동안 성능 향상이 없으면 조기 종료
VAL_RATIO = 0.2             # 검증(val) 데이터 비율
SEED = 42
RUN_NAME = "yolo26_baseball_pose"
# ==============================================

RAW_DIR = f"{WORK_DIR}/Baseball_Sample"            # 원본 압축 해제 위치
DATASET_DIR = f"{WORK_DIR}/baseball_dataset"       # YOLO 형식 변환 결과
DATA_YAML = f"{DATASET_DIR}/baseball_data.yaml"
RUNS_DIR = f"{DRIVE_DIR}/runs"

## 2. GPU 확인 · 구글 드라이브 연결 · 라이브러리 설치

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount("/content/drive")

!pip install -q -U ultralytics
import ultralytics
ultralytics.checks()

## 3. 원본 데이터 압축 해제

Windows에서 만든 zip은 한글 폴더명이 깨질 수 있어서, 파일명을 cp949로 복원하면서 풉니다.

In [ ]:
import os, shutil, zipfile

shutil.rmtree(RAW_DIR, ignore_errors=True)
os.makedirs(RAW_DIR, exist_ok=True)

with zipfile.ZipFile(SAMPLE_ZIP) as zf:
    for info in zf.infolist():
        name = info.filename
        # UTF-8 플래그가 없으면 Windows(cp949)로 압축된 한글 이름 -> 복원
        if not info.flag_bits & 0x800:
            try:
                name = name.encode("cp437").decode("cp949")
            except UnicodeError:
                pass
        target = os.path.join(RAW_DIR, name)
        if name.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with zf.open(info) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

!find "{RAW_DIR}" -maxdepth 3 -type d

## 4. AIHUB JSON → YOLO Pose 형식 변환

- 라벨 한 줄: `클래스 cx cy w h  x1 y1 v1 ... x24 y24 v24` (모두 0~1로 정규화)
- 같은 영상(클립)의 프레임은 거의 똑같으므로 **클립 단위로 train/val 분할** (프레임 단위로 섞으면 val 점수가 부풀려짐)

In [ ]:
import glob, json, random
import yaml

# 이전 변환 결과를 지우고 images/labels 의 train, val 폴더를 새로 만듭니다.
shutil.rmtree(DATASET_DIR, ignore_errors=True)
for sub in ("images", "labels"):
    for split in ("train", "val"):
        os.makedirs(f"{DATASET_DIR}/{sub}/{split}", exist_ok=True)

# 원천 이미지 파일명 -> 전체 경로 매핑
image_paths = {os.path.basename(p): p for p in glob.glob(f"{RAW_DIR}/**/*.jpg", recursive=True)}
json_files = sorted(glob.glob(f"{RAW_DIR}/**/*.json", recursive=True))
print(f"JSON {len(json_files)}개, 이미지 {len(image_paths)}개")

# 클립(JSON 이 들어있는 폴더) 단위로 train/val 분할
clips = sorted({os.path.basename(os.path.dirname(p)) for p in json_files})
random.Random(SEED).shuffle(clips)
num_val = max(1, int(len(clips) * VAL_RATIO))
val_clips = set(clips[:num_val])
print(f"클립 {len(clips)}개 -> train {len(clips) - num_val}개 / val {num_val}개")

counts = {"train": 0, "val": 0}
keypoint_names = None

for json_path in json_files:
    try:
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        keypoint_names = keypoint_names or data["categories"]["keypoints"]
        img_w, img_h = data["image"]["resolution"]

        # 바운딩 박스와 키포인트 추출 (AIHUB 는 bbox / points 가 별도 annotation 으로 들어있음)
        bboxes = [a["bbox"] for a in data["annotations"] if "bbox" in a]      # [x_min, y_min, x_max, y_max]
        points = [a["points"] for a in data["annotations"] if "points" in a]  # [x1, y1, v1, x2, y2, v2, ...]
        if not bboxes or not points:
            continue

        image_filename = data["image"]["filename"]
        image_path = image_paths.get(image_filename)
        if image_path is None:
            print(f"이미지 없음, 건너뜀: {image_filename}")
            continue

        lines = []
        for (x_min, y_min, x_max, y_max), kps in zip(bboxes, points):
            # 박스: 중심 x, 중심 y, 너비, 높이 (0~1 정규화)
            box = [(x_min + x_max) / 2 / img_w, (y_min + y_max) / 2 / img_h,
                   (x_max - x_min) / img_w, (y_max - y_min) / img_h]
            # 키포인트: x, y 정규화 / 가시성 v(0: 없음, 1: 가려짐, 2: 보임)는 그대로
            kp = []
            for i in range(0, len(kps), 3):
                kp += [f"{kps[i] / img_w:.6f}", f"{kps[i + 1] / img_h:.6f}", str(int(kps[i + 2]))]
            lines.append("0 " + " ".join(f"{v:.6f}" for v in box) + " " + " ".join(kp))

        split = "val" if os.path.basename(os.path.dirname(json_path)) in val_clips else "train"
        shutil.copy2(image_path, f"{DATASET_DIR}/images/{split}/{image_filename}")
        with open(f"{DATASET_DIR}/labels/{split}/{os.path.splitext(image_filename)[0]}.txt", "w") as f:
            f.write("\n".join(lines) + "\n")
        counts[split] += 1
    except Exception as e:
        print(f"변환 오류 ({os.path.basename(json_path)}): {e}")

print(f"변환 완료: train {counts['train']}장 / val {counts['val']}장")

# 좌우 반전 증강 시 서로 바뀌어야 할 관절 번호 (right_xxx <-> left_xxx)
# 없으면 YOLO 가 좌우 반전 증강을 자동으로 끕니다.
def mirror(name):
    if name.startswith("right_"): return "left_" + name[6:]
    if name.startswith("left_"): return "right_" + name[5:]
    if name.endswith("_right"): return name[:-6] + "_left"
    if name.endswith("_left"): return name[:-5] + "_right"
    return name
flip_idx = [keypoint_names.index(mirror(n)) for n in keypoint_names]

data_cfg = {
    "path": DATASET_DIR,
    "train": "images/train",
    "val": "images/val",
    "names": {0: "person"},
    "kpt_shape": [len(keypoint_names), 3],   # AIHUB 야구: 24개 관절 x (x, y, 가시성)
    "flip_idx": flip_idx,
    "kpt_names": {0: keypoint_names},
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, allow_unicode=True, sort_keys=False)

print(open(DATA_YAML, encoding="utf-8").read())

## 5. 라벨 확인 (학습 전 눈으로 검증)

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample = sorted(glob.glob(f"{DATASET_DIR}/images/train/*.jpg"))[0]
img = cv2.cvtColor(cv2.imread(sample), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
label = open(sample.replace("/images/", "/labels/").replace(".jpg", ".txt")).readline().split()
cx, cy, bw, bh = [float(v) for v in label[1:5]]
kps = [float(v) for v in label[5:]]

plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.gca().add_patch(plt.Rectangle(((cx - bw / 2) * w, (cy - bh / 2) * h), bw * w, bh * h, fill=False, color="lime", lw=2))
for i in range(0, len(kps), 3):
    if kps[i + 2] > 0:
        plt.scatter(kps[i] * w, kps[i + 1] * h, s=15, c="red" if kps[i + 2] == 2 else "orange")
plt.title(os.path.basename(sample) + "  (red: visible / orange: occluded)")
plt.axis("off")
plt.show()

## 6. 학습

결과는 구글 드라이브(`RUNS_DIR`)에 바로 저장되므로 런타임이 끊겨도 남아 있습니다.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)   # COCO 로 사전학습된 YOLO26 Pose 가중치에서 시작 (전이학습)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    workers=2,          # Colab 은 CPU 가 2개
    seed=SEED,
    project=RUNS_DIR,
    name=RUN_NAME,
    exist_ok=True,
)
RUN_DIR = str(model.trainer.save_dir)
print("학습 결과 폴더:", RUN_DIR)

### (필요할 때만) 끊긴 학습 이어서 하기

런타임이 끊겼다면 **1~4번 셀을 다시 실행한 뒤** 아래 셀을 실행하세요.

In [ ]:
RUN_DIR = f"{RUNS_DIR}/{RUN_NAME}"
model = YOLO(f"{RUN_DIR}/weights/last.pt")
model.train(resume=True)

## 7. 성능 평가 (val)

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO

RUN_DIR = f"{RUNS_DIR}/{RUN_NAME}"
BEST_PT = f"{RUN_DIR}/weights/best.pt"

best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split="val")
print(f"Pose mAP50    : {metrics.pose.map50:.4f}")
print(f"Pose mAP50-95 : {metrics.pose.map:.4f}")
print(f"Box  mAP50-95 : {metrics.box.map:.4f}")

display(Image(f"{RUN_DIR}/results.png", width=900))

## 8. 검증 이미지로 예측 결과 보기

In [ ]:
val_images = sorted(glob.glob(f"{DATASET_DIR}/images/val/*.jpg"))
for path in random.Random(0).sample(val_images, k=min(3, len(val_images))):
    result = best.predict(path, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    plt.figure(figsize=(12, 7))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.title(os.path.basename(path))
    plt.axis("off")
    plt.show()

## 9. 모델 저장 · 다운로드

`best_baseball_pose.pt` 를 로컬 프로젝트의 `AI-Server` 에 넣어 사용합니다.

In [ ]:
FINAL_PT = f"{DRIVE_DIR}/best_baseball_pose.pt"
shutil.copy2(BEST_PT, FINAL_PT)
print("드라이브에 저장:", FINAL_PT)

from google.colab import files
files.download(FINAL_PT)